In [1]:
!pip install pandas numpy scikit-learn streamlit pyngrok

  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
   ---------------------------------------- 0.0/9.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.1 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.1 MB 3.3 MB/s eta 0:00:03
   ------ --------------------------------- 1.6/9.1 MB 4.0 MB/s eta 0:00:02
   --------------- ------------------------ 3.4/9.1 MB 6.5 MB/s eta 0:00:01
   --------------- ------------------------ 3.4/9.1 MB 6.5 MB/s eta 0:00:01
   --------------- ------------------------ 3.4/9.1 MB 6.5 MB/s eta 0:00:01
   ---------------------- ----------------- 5.0/9.1 MB 4.0 MB/s eta 0:00:02
   --------------------------------- ------ 7.6/9.1 MB 5.4 MB/s eta 0:00:01
   ------------------------------------- -- 8.4/9.1 MB 5.1 MB/s eta 0:00:01
   ---------------------------------------- 9.1/9.1 MB 5.0 MB/s eta 0:00:00
   ---------------------------------------- 0.0/797.0 kB ? eta -:--:--
   -----------------------------

In [17]:
import pandas as pd
import zipfile
import requests
import io

url = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"

r = requests.get(url)
z = zipfile.ZipFile(io.BytesIO(r.content))

# Extract and load
movies = pd.read_csv(z.open("ml-latest-small/movies.csv"))
movies = movies.rename(columns={"genres": "feature"})
movies.head()

,movieId,title,feature
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [13]:
books = pd.DataFrame({
    "id":[1,2,3,4],
    "title":["The Hobbit","Harry Potter","Da Vinci Code","Pride and Prejudice"],
    "feature":["Fantasy","Fantasy","Thriller","Romance"]
})

In [14]:
restaurants = pd.DataFrame({
    "id":[1,2,3,4],
    "name":["Dominos","Spice Hub","Sushi Place","Olive Garden"],
    "feature":["Italian","Indian","Japanese","Italian"]
})

In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

class Recommender:
    def __init__(self, df):
        self.df = df
        self.vectorizer = TfidfVectorizer(stop_words='english')
        self.matrix = self.vectorizer.fit_transform(df['feature'])

    def recommend(self, index, top_n=5):
        sim = cosine_similarity(self.matrix[index], self.matrix).flatten()
        idx = sim.argsort()[::-1][1:top_n+1]
        return self.df.iloc[idx]

In [19]:
movie_model = Recommender(movies)
book_model = Recommender(books)
restaurant_model = Recommender(restaurants)

In [20]:
def cross_domain(movie_index):
    movie = movies.iloc[movie_index]
    genre = movie['feature']

    book_rec = books[books['feature'].str.contains(genre.split("|")[0], na=False)]
    
    if "Romance" in genre:
        rest = restaurants[restaurants['feature']=="Italian"]
    elif "Action" in genre:
        rest = restaurants.sample(2)
    else:
        rest = restaurants.sample(2)

    return {
        "movie": movie['title'],
        "books": book_rec['title'].tolist(),
        "restaurants": rest['name'].tolist()
    }

In [21]:
cross_domain(10)

{'movie': 'American President, The (1995)',
 'books': [],
 'restaurants': ['Dominos', 'Olive Garden']}